# Agent Architectures: ReAct, Plan-and-Execute and Reflection, Compared

The runnable code of [Agent Architectures: ReAct, Plan-and-Execute and Reflection, Compared](https://residual-stream.lvtan71.workers.dev/posts/agent-architectures-react-plan-reflection/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/agent-architectures-react-plan-reflection.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 87 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "agent-architectures-react-plan-reflection"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## The three, measured

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/agent-architectures-react-plan-reflection/#the-three-measured)

In [ ]:
import json
import random
import re
import time
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"

# Three agent architectures on the same tasks, model and tools.
# The world: 12 fictional companies with founding years, headcounts and
# cities, so nothing can be answered from memory. Tools: lookup(company,
# field) and calculator(expression). Each question needs 2-3 lookups and
# some arithmetic. Model: Qwen2.5-3B-Instruct, temperature 0.7, 3 runs each.
#   ReAct: think and act in one loop until it answers.
#   Plan-and-execute: write a numbered plan first (no tools), then carry out
#     each step in its own short tool loop, passing on each call made and
#     its result, verbatim; then a final tool loop answers from them.
#   Reflection: ReAct, then a critique of the transcript; if the critic
#     finds a problem, one more ReAct attempt with the critique attached.
# A reply counts as an answer only if it contains "Answer: <number>".
PATH = snapshot_download("Qwen/Qwen2.5-3B-Instruct",
                         revision="aa8e72537993ba99e69dfaafa59ed015b17504d1")
tok = AutoTokenizer.from_pretrained(PATH)
model = AutoModelForCausalLM.from_pretrained(PATH, dtype=torch.float16).to(GPU).eval()

rng = random.Random(1)
names = ["Arvelo", "Brindle", "Cobaltic", "Dunmore", "Elstree", "Farrow", "Gilden", "Harrowgate", "Istria",
         "Juniper Labs", "Kestrel", "Lumen Forge"]
cities = ["Lyon", "Osaka", "Porto", "Tartu", "Quito", "Perth"]
db = {n: {"founded": rng.randint(1950, 2020), "employees": rng.randint(40, 9000), "city": rng.choice(cities)}
      for n in names}
TOOLS = [{"type": "function", "function": {"name": "lookup", "description": "Look up one field of a company: "
          "the year it was founded, its number of employees, or its city.", "parameters": {"type": "object", "properties": {
          "company": {"type": "string"}, "field": {"type": "string", "enum": ["founded", "employees", "city"]}},
          "required": ["company", "field"]}}},
         {"type": "function", "function": {"name": "calculator", "description": "Evaluate an arithmetic expression.",
          "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}}]
def tool(name, args):
    if name == "lookup":                      # errors say exactly what would have worked
        company, field = str(args.get("company", "")).strip(), args.get("field")
        if company not in db:
            return f"error: no company named {company!r}; companies are: {', '.join(names)}"
        if field not in db[company]:
            return f"error: no field {field!r}; fields are: founded, employees, city"
        return str(db[company][field])
    if name == "calculator":
        e = str(args.get("expression", ""))
        return str(round(eval(e), 4)) if re.fullmatch(r"[\d\s+\-*/().]+", e) else "error: arithmetic only"
    return "error: unknown tool"

tasks = []
for _ in range(20):
    a, b, c = rng.sample(names, 3)
    kind = rng.randrange(3)
    if kind == 0:
        tasks.append((f"How many years older is {a} than {b}? (Use founding years; negative if younger.)",
                      db[b]["founded"] - db[a]["founded"]))
    elif kind == 1:
        tasks.append((f"How many employees do {a}, {b} and {c} have in total?",
                      db[a]["employees"] + db[b]["employees"] + db[c]["employees"]))
    else:
        tasks.append((f"How many more employees does {a} have than {b}? (Negative if fewer.)",
                      db[a]["employees"] - db[b]["employees"]))

SYS = "You answer questions using the tools. Never guess a value; look it up. End with: Answer: <number>"
@torch.no_grad()
def gen(messages, tools=TOOLS, max_new=384):
    ids = tok.apply_chat_template(messages, tools=tools, add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True)["input_ids"].to(GPU)
    out = model.generate(ids, max_new_tokens=max_new, do_sample=True, temperature=0.7, top_p=1.0, top_k=0)
    gen.tokens += out.shape[1]                # tokens processed: prompt + output, every call
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

def tool_loop(messages, max_calls=8):
    """Run model calls, executing tool calls, until a reply without one."""
    tool_loop.calls = []                      # each executed call with its result, for plan-and-execute
    for _ in range(max_calls):
        text = gen(messages)
        messages.append({"role": "assistant", "content": text})
        calls = re.findall(r"<tool_call>\s*(\{.*?)\s*</tool_call>", text, re.S)
        if not calls:
            return text
        for c in calls:
            try:                                  # tolerate trailing junk such as an extra "}"
                call = json.JSONDecoder().raw_decode(c)[0]; r = tool(call["name"], call.get("arguments", {}))
                tool_loop.calls.append(f"{call['name']}({json.dumps(call.get('arguments', {}))}) = {r}")
            except (ValueError, KeyError, TypeError, AttributeError):
                r = ('error: could not parse the tool call; write exactly <tool_call>{"name": "lookup", '
                     '"arguments": {"company": "...", "field": "employees"}}</tool_call>')
            messages.append({"role": "tool", "content": r})
    return ""

def react(q):
    return tool_loop([{"role": "system", "content": SYS}, {"role": "user", "content": q}])

def plan_execute(q):
    plan = gen([{"role": "user", "content": "Write a short numbered plan (one tool action per line, no answers) "
                 f"for answering this with lookup and calculator tools:\n{q}"}], tools=None, max_new=200)
    steps = [l for l in plan.splitlines() if re.match(r"\s*(\d+[.):]|[-*]|step\s*\d+)", l, re.I)][:6]
    steps = steps or [l for l in plan.splitlines() if l.strip()][:6]
    LOG["plan"] = steps
    notes = []                                # each step's tool calls and results, verbatim
    for s in steps:
        msgs = [{"role": "system", "content": "Do exactly this one step with the tools."},
                {"role": "user", "content": f"Question: {q}\nResults so far: {notes}\nStep: {s}"}]
        tool_loop(msgs, max_calls=3)
        notes += tool_loop.calls
    LOG["notes"] = notes
    return tool_loop([{"role": "system", "content": SYS}, {"role": "user", "content":
                      f"{q}\nTool results from the plan's steps: {notes}"}])

def reflect(q):
    msgs = [{"role": "system", "content": SYS}, {"role": "user", "content": q}]
    first = tool_loop(msgs)
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in msgs[1:])[-3000:]
    critique = gen([{"role": "user", "content": "Check this attempt. Were all values looked up (not guessed) and is the "
                     f"arithmetic right? Reply OK, or describe the error.\n\n{transcript}"}], tools=None, max_new=150)
    LOG["critique"] = critique
    if re.match(r"\W*OK\b", critique.strip().upper()):
        return first
    return tool_loop([{"role": "system", "content": SYS}, {"role": "user", "content":
                       f"{q}\nA previous attempt was checked and this problem was found: {critique}"}])

def answer(text):
    m = re.search(r"Answer:\s*(-?[\d,]+)", text or "")
    return int(m.group(1).replace(",", "")) if m else None

print(f"{len(tasks)} questions x 3 runs; Qwen2.5-3B-Instruct\n")
print(f"{'architecture':18s} {'right':>6s} {'tokens processed/question':>26s} {'seconds/question':>17s}")
import os
LOG = {}
done = {}                                     # each trial is logged, so a stopped run resumes
if os.path.exists("arch_trials_v3.jsonl"):
    for l in open("arch_trials_v3.jsonl"):
        t = json.loads(l); done[(t["arch"], t["i"], t["k"])] = t
for name, f in (("ReAct", react), ("plan-and-execute", plan_execute), ("reflection", reflect)):
    right, toks, secs = [], [], []
    for i, (q, truth) in enumerate(tasks):
        for k in range(3):
            if (name, i, k) not in done:
                torch.manual_seed(1000 * i + k); gen.tokens = 0; t0 = time.perf_counter(); LOG.clear()
                reply = f(q)
                t = {"arch": name, "i": i, "k": k, "pred": answer(reply), "truth": truth, "tokens": gen.tokens,
                     "secs": time.perf_counter() - t0, "reply": reply[-600:], **LOG}
                with open("arch_trials_v3.jsonl", "a") as fh:
                    fh.write(json.dumps(t) + "\n")
                done[(name, i, k)] = t
            t = done[(name, i, k)]
            right.append(t["pred"] == truth); toks.append(t["tokens"]); secs.append(t["secs"])
    print(f"{name:18s} {np.mean(right):6.0%} {np.mean(toks):26,.0f} {np.mean(secs):17.1f}", flush=True)

The post printed:

```text
20 questions x 3 runs; Qwen2.5-3B-Instruct

architecture        right  tokens processed/question  seconds/question
ReAct                 35%                      1,176              12.1
plan-and-execute      48%                      6,058              51.7
reflection            37%                      2,420              29.6
```

In [ ]:
import collections
import itertools
import json
import math
import re
import random

# Why each wrong answer was wrong. Rebuilds the same companies and questions
# as arch.py (same seed), then sorts every logged trial into: right; the
# sign flipped; a total missing one company; no answer; other.
rng = random.Random(1)
names = ["Arvelo", "Brindle", "Cobaltic", "Dunmore", "Elstree", "Farrow", "Gilden", "Harrowgate", "Istria",
         "Juniper Labs", "Kestrel", "Lumen Forge"]
cities = ["Lyon", "Osaka", "Porto", "Tartu", "Quito", "Perth"]
db = {n: {"founded": rng.randint(1950, 2020), "employees": rng.randint(40, 9000), "city": rng.choice(cities)}
      for n in names}
kinds, parts = [], []
for _ in range(20):
    a, b, c = rng.sample(names, 3)
    kind = rng.randrange(3)
    kinds.append(["years older", "total of three", "more employees"][kind])
    parts.append([db[x]["employees"] for x in (a, b, c)] if kind == 1 else [])

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return f"{k / n:4.0%} [{c - h:.0%}, {c + h:.0%}]"

trials = [json.loads(l) for l in open("arch_trials_v3.jsonl")]
archs = list(dict.fromkeys(t["arch"] for t in trials))
print(f"{'':18s} {'right [95% CI]':>18s} {'sign flipped':>13s} {'total missing one':>18s} {'no answer':>10s} {'other':>6s}")
for a in archs:
    c = collections.Counter()
    ts = [t for t in trials if t["arch"] == a]
    for t in ts:
        p, y, i = t["pred"], t["truth"], t["i"]
        if p == y: c["right"] += 1
        elif p is None: c["none"] += 1
        elif p == -y: c["sign"] += 1
        elif parts[i] and p in {x + z for x, z in itertools.combinations(parts[i], 2)}: c["missing"] += 1
        else: c["other"] += 1
    print(f"{a:18s} {wilson(c['right'], len(ts)):>18s} {c['sign']:>13d} {c['missing']:>18d} {c['none']:>10d} {c['other']:>6d}")
print("\nright, by question type:")
for a in archs:
    ts = [t for t in trials if t["arch"] == a]
    row = []
    for k in ("years older", "more employees", "total of three"):
        sub = [t for t in ts if kinds[t["i"]] == k]
        row.append(f"{k} {sum(t['pred'] == t['truth'] for t in sub)}/{len(sub)}")
    print(f"  {a:18s} " + ", ".join(row))

# Paired: same question, same run index, two architectures.
print("\npaired (same question and run): only the first right / only the second right")
ok = {(t["arch"], t["i"], t["k"]): t["pred"] == t["truth"] for t in trials}
for a, b in itertools.combinations(archs, 2):
    keys = [(i, k) for (x, i, k) in ok if x == a and (b, i, k) in ok]
    print(f"  {a} vs {b}: {sum(ok[(a, *q)] and not ok[(b, *q)] for q in keys)} / "
          f"{sum(ok[(b, *q)] and not ok[(a, *q)] for q in keys)}")

# Reflection: its first attempt is ReAct's run (same seed, same prompt),
# and the critic's reply is logged, so each trial shows what it did.
r = {(t["i"], t["k"]): t for t in trials if t["arch"] == "ReAct"}
f = [t for t in trials if t["arch"] == "reflection"]
flagged = [t for t in f if not re.match(r"\W*OK\b", t["critique"].strip().upper())]
first_ok = lambda t: r[(t["i"], t["k"])]["pred"] == t["truth"]
c = collections.Counter((first_ok(t), t["pred"] == t["truth"]) for t in flagged)
print(f"\nreflection: the critic flagged {len(flagged)} of {len(f)} first attempts "
      f"({sum(map(first_ok, flagged))} of the {sum(map(first_ok, f))} right ones, "
      f"{len(flagged) - sum(map(first_ok, flagged))} of the {len(f) - sum(map(first_ok, f))} wrong ones)")
print(f"  after the retry: wrong->right {c[(False, True)]}, right->wrong {c[(True, False)]}, "
      f"wrong->wrong {c[(False, False)]}, right->right {c[(True, True)]}")
signs = [t for t in f if r[(t["i"], t["k"])]["pred"] == -t["truth"]]
print(f"  first attempts with the sign flipped: {len(signs)}; flagged {sum(t in flagged for t in signs)}; "
      f"right after the retry {sum(t['pred'] == t['truth'] for t in signs)}")

The post printed:

```text
                       right [95% CI]  sign flipped  total missing one  no answer  other
ReAct                  35% [24%, 48%]            17                  9          1     12
reflection             37% [26%, 49%]            19                  4          8      7
plan-and-execute       48% [36%, 61%]            22                  2          0      7

right, by question type:
  ReAct              years older 1/21, more employees 16/21, total of three 4/18
  reflection         years older 2/21, more employees 13/21, total of three 7/18
  plan-and-execute   years older 0/21, more employees 18/21, total of three 11/18

paired (same question and run): only the first right / only the second right
  ReAct vs reflection: 5 / 6
  ReAct vs plan-and-execute: 4 / 12
  reflection vs plan-and-execute: 5 / 12

reflection: the critic flagged 44 of 60 first attempts (16 of the 21 right ones, 28 of the 39 wrong ones)
  after the retry: wrong->right 6, right->wrong 5, wrong->wrong 22, right->right 11
  first attempts with the sign flipped: 17; flagged 11; right after the retry 1
```